In [1]:
import tensorflow as tf

In [2]:
file_url = 'https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip'

In [3]:
zip_dir = tf.keras.utils.get_file('pizza_and_streak.zip', origin=file_url, extract=True)

109540975/109540975 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [4]:
import pathlib

In [5]:
path = pathlib.Path(zip_dir).parent / 'pizza_steak'

In [6]:
train_dir = path / 'train'
validation_dir = path / 'test'

In [12]:
train_dir

PosixPath('/root/.keras/datasets/pizza_steak/train')

In [13]:
train_pizza_dir = train_dir / 'pizza'
train_steak_dir = train_dir /'steak'
validation_pizza_dir = validation_dir / 'pizza'
validation_steak_dir = validation_dir / 'steak'

In [14]:
import os

In [15]:
total_train = len(os.listdir(train_pizza_dir)) + len(os.listdir(train_steak_dir))
total_val = len(os.listdir(validation_pizza_dir)) + len(os.listdir(validation_steak_dir))

In [16]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [17]:
train_image_generator = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)

In [18]:
batch_size = 16
img_height = 224
img_width = 224

In [19]:
train_data_gen = train_image_generator.flow_from_directory(batch_size = batch_size,
                                                           directory = train_dir,
                                                           shuffle=True,
                                                           target_size = (img_height, img_width),
                                                           class_mode='binary')

Found 1500 images belonging to 2 classes.


In [20]:
val_data_gen = validation_image_generator.flow_from_directory(batch_size = batch_size,
                                                              directory = validation_dir,
                                                              target_size=(img_height, img_width),
                                                              class_mode='binary')

Found 500 images belonging to 2 classes.


In [21]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [22]:
np.random.seed(8)
tf.random.set_seed(8)

In [23]:
from tensorflow.keras.applications import NASNetMobile

In [25]:
base_model = NASNetMobile(include_top=False,
                          input_shape=(img_height, img_width, 3),
                          weights='imagenet')

In [26]:
base_model.trainable = False

In [27]:
base_model.summary()

Model: "nasnet_mobile"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv1 (Conv2D)       │ (None, 111, 111, 32)   │            864 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn1                  │ (None, 111, 111, 32)   │            128 │ stem_conv1[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_188            │ (None, 111, 111, 32)   │              0 │ stem_bn1[0][0]         │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reduction_conv_1_stem_1   │ (None, 111, 111, 11)   │            352 │ activation_188[0][0]   │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reduction_bn_1_stem_1     │ (None, 111, 111, 11)   │             44 │ reduction_conv_1_stem… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_189            │ (None, 111, 111, 11)   │              0 │ reduction_bn_1_stem_1… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_191            │ (None, 111, 111, 32)   │              0 │ stem_bn1[0][0]         │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_pad_red… │ (None, 115, 115, 11)   │              0 │ activation_189[0][0]   │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_pad_red… │ (None, 117, 117, 32)   │              0 │ activation_191[0][0]   │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_reducti… │ (None, 56, 56, 11)     │            396 │ separable_conv_1_pad_… │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_reducti… │ (None, 56, 56, 11)     │          1,920 │ separable_conv_1_pad_… │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_bn_redu… │ (None, 56, 56, 11)     │             44 │ separable_conv_1_redu… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ separable_conv_1_bn_r

 Total params: 4,269,716 (16.29 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,269,716 (16.29 MB)

In [28]:
model = tf.keras.Sequential([base_model,layers.Flatten(),
                             layers.Dense(500, activation='relu'),
                             layers.Dense(1, activation='sigmoid')])

In [29]:
model.compile(loss='binary_crossentropy',
              optimizer=tf.keras.optimizers.Adam(0.001),
              metrics=['accuracy'])

In [30]:
model.fit(train_data_gen,
          steps_per_epoch = total_train // batch_size,
          epochs=5,
          validation_data = val_data_gen,
          validation_steps = total_val // batch_size)

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


93/93 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.9092 - loss: 2.0208 - val_accuracy: 0.9879 - val_loss: 0.2707
Epoch 2/5
 1/93 ━━━━━━━━━━━━━━━━━━━━ 2:33 2s/step - accuracy: 1.0000 - loss: 0.0011

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 1.2667e-35
Epoch 3/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 242s 2s/step - accuracy: 0.9747 - loss: 0.4620 - val_accuracy: 0.9637 - val_loss: 1.4524
Epoch 4/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 39s 403ms/step - accuracy: 0.9375 - loss: 2.9924 - val_accuracy: 1.0000 - val_loss: 1.0002e-33
Epoch 5/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.9617 - loss: 0.8774 - val_accuracy: 0.9839 - val_loss: 0.5651
